In [ ]:
import os
import psycopg2
import pandas as pd
import boto3
from dotenv import load_dotenv
from sklearn.impute import SimpleImputer

#Chargement des identifiants depuis le fichier d'environnement (bloqué sur github par .gitignore de *.env)
load_dotenv("credential.env")

# Connexion à notre base de données PostgreSQL hébergée sur AWS RDS
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

# Configuration du client S3 pour aller chercher nos données sources
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name="eu-west-3"
)

# Paramètres de connexion au bucket 'certification-kayak' fichier kayak_ds.csv
nom_bucket = os.getenv("AWS_BUCKET_NAME")  
cle_s3 = "kayak_ds.csv"

# Récupération du fichier (kayak_ds.csv) qui est sur le datalake ('certification-kayak')
print('Récupération du fichier S3 kayak_ds.csv')
response = s3_client.get_object(Bucket=nom_bucket, Key=cle_s3)
df = pd.read_csv(response['Body'], encoding="utf-8-sig")


# Gestion des valeurs manquantes
    # Remplacement par la moyenne pour éviter les plantage de la base SQL !=nan
imputer = SimpleImputer(strategy="mean")

# La colonne 'note' a des valeurs nan, remplacement par la note moyenne -> 8.5 (arrondi à 1 décimale)
df['note'] = imputer.fit_transform(df[['note']]).round(1)


# Instanciation d'un curseur pour interagir avec la base sql
cursor = conn.cursor()

# Création de la table 'kayak_data' si elle n'existe pas déjà
# Typage des colonnes en fonction du contenu
print("création de la table 'kayak_data' sur Amazon RDS")
cursor.execute("""
    DROP TABLE IF EXISTS kayak_data;
    CREATE TABLE kayak_data (
        id INT,
        ville VARCHAR(100),
        latitude DOUBLE PRECISION,
        longitude DOUBLE PRECISION,
        beau_temps NUMERIC(5,2),
        temperature NUMERIC(4,1),
        pluie NUMERIC(5,2),
        humidite INT,
        periode VARCHAR(50),
        nom_hotel VARCHAR(255),
        url_hotel TEXT,
        note NUMERIC(3,1)
    );
""")
# On valide la création de la table dans la base
conn.commit()

# Création de la structure de la requête sql 
# Chaque %s va être remplacé par la valeur de chaque ligne du dataframe et ce dans l'ordre de chaque colonne
query = """
    INSERT INTO kayak_data (id, ville, latitude, longitude, beau_temps, temperature, pluie, humidite, periode, nom_hotel, url_hotel, note)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
"""

# On parcourt chaque ligne du dataframe pour l'envoyer vers la base
for valeurs in df.itertuples(index=False, name=None):
    # La base postgres est alimentée ligne par ligne
    cursor.execute(query, valeurs)

# On confirme que tous les inserts sont bien enregistrés
conn.commit()
print("La base kayak_data est finie d'être alimentée")

print('\nContrôle de la base')
# Nombre de lignes total dans notre nouvelle table
cursor.execute("SELECT COUNT(*) FROM kayak_data;")
print(f"Lignes sur kayak_data : {cursor.fetchone()[0]}")

# On vérifie avec une ligne d'exemple que tout est bien formaté
cursor.execute("SELECT * FROM kayak_data LIMIT 1;")
print(f"Exemple : {cursor.fetchone()}")

# On ferme la communication avec la base pour libérer les ressources
cursor.close()
conn.close()

g:\prog_data_scientist\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Récupération du fichier S3 kayak_ds.csv
création de la table 'kayak_data' sur Amazon RDS
La base kayak_data est finie d'être alimentée

Contrôle de la base
Lignes sur kayak_data : 700
Exemple : (1, 'Mont Saint Michel', 48.6359541, -1.51146, Decimal('109.30'), Decimal('26.9'), Decimal('0.37'), 82, 'du 24/06/2026 au 01/07/2026', 'Hôtel Vert', 'https://www.booking.com/hotel/fr/vert.fr.html?aid=304142&label=gen173nr-10CAQoggJCGHNlYXJjaF9tb250IHNhaW50IG1pY2hlbEgNWARoTYgBAZgBM7gBF8gBD9gBA-gBAfgBAYgCAagCAbgC18ju0QbAAgHSAiQ3ZThjYTJiYS1iNWIxLTRmYjktYTc4Mi1jYmQwN2M1NGQ4M2LYAgHgAgE&ucfs=1&arphpl=1&group_adults=2&req_adults=2&no_rooms=1&group_children=0&req_children=0&hpos=1&hapos=1&sr_order=popularity&srpvid=a411432b5ef4020c&srepoch=1782293593&from=searchresults', Decimal('8.2'))
